In [27]:
# Load env variables and create client
from dotenv import load_dotenv
import os 
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = os.environ["CLAUDE_MODEL"] 

In [28]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [29]:
import json

def generate_dataset():  
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json") 
    text = chat(messages, stop_sequences= ["```"]) 
    return json.loads(text) 

In [30]:
dataset = generate_dataset() 
dataset

[{'task': 'Write a Python function that parses an AWS ARN string and returns a dictionary containing the partition, service, region, account-id, and resource components.'},
 {'task': "Create a JSON object for an AWS IAM policy that allows read-only access to a specific S3 bucket named 'my-data-bucket' for objects with the prefix 'reports/'."},
 {'task': "Write a regular expression that validates AWS EC2 instance IDs, which start with 'i-' followed by either 8 or 17 hexadecimal characters (e.g., i-1234abcd or i-0123456789abcdef0)."}]

In [31]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent = 2) 

In [32]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
        Please solve the following task:

        {test_case["task"]} 
    """
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages) 
    return output 

In [39]:
# Graders:
#     1. Code - Programatically evaluate the result 
#     2. Model - Ask a model to assign a score to the output or compare two versions
#     3. Human - Ask a Human to assign a score to the output, or compare two versions      


def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
        You are an expert code reviewer. Evaluate this AI-generated solution.
        
        Task: {test_case}
        Solution: {output}
        
        Provide your evaluation as a structured JSON object with:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement  
        - "reasoning": A concise explanation of your assessment
        - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text) 

In [40]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [41]:
from statistics import mean 
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results 

In [42]:
with open("dataset.json", "r") as f:
    dataset = json.load(f) 

results = run_eval(dataset) 

Average score: 7.333333333333333


In [43]:
results 

[{'output': 'I\'ll help you write a Python function to parse AWS ARN strings.\n\n```python\ndef parse_arn(arn_string):\n    """\n    Parses an AWS ARN string and returns its components as a dictionary.\n    \n    ARN Format: arn:partition:service:region:account-id:resource\n    or: arn:partition:service:region:account-id:resourcetype/resource\n    or: arn:partition:service:region:account-id:resourcetype:resource\n    \n    Args:\n        arn_string (str): The ARN string to parse\n        \n    Returns:\n        dict: A dictionary containing the ARN components\n        \n    Raises:\n        ValueError: If the ARN string is invalid\n    """\n    if not arn_string or not isinstance(arn_string, str):\n        raise ValueError("ARN must be a non-empty string")\n    \n    # Split the ARN by colons\n    parts = arn_string.split(\':\', 5)  # Split into max 6 parts\n    \n    # Validate that we have at least 6 parts\n    if len(parts) != 6 or parts[0] != \'arn\':\n        raise ValueError(f"In

In [44]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "I'll help you write a Python function to parse AWS ARN strings.\n\n```python\ndef parse_arn(arn_string):\n    \"\"\"\n    Parses an AWS ARN string and returns its components as a dictionary.\n    \n    ARN Format: arn:partition:service:region:account-id:resource\n    or: arn:partition:service:region:account-id:resourcetype/resource\n    or: arn:partition:service:region:account-id:resourcetype:resource\n    \n    Args:\n        arn_string (str): The ARN string to parse\n        \n    Returns:\n        dict: A dictionary containing the ARN components\n        \n    Raises:\n        ValueError: If the ARN string is invalid\n    \"\"\"\n    if not arn_string or not isinstance(arn_string, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    # Split the ARN by colons\n    parts = arn_string.split(':', 5)  # Split into max 6 parts\n    \n    # Validate that we have at least 6 parts\n    if len(parts) != 6 or parts[0] != 'arn':\n        raise Valu